# 7.11 Better CNN Practice

[Back to neural networks index](../7_neural_networks.ipynb)

This notebook moves beyond the first convolutional classifier and focuses on the workflow habits that make CNN training more reliable and interpretable.


## 7.11.1 Normalization

### Why It Matters
Image models usually benefit from inputs that are centered and scaled consistently. Here we compare raw and normalized pixel summaries.


In [ ]:
import torch
from sklearn.datasets import load_digits

X = torch.tensor(load_digits().images, dtype=torch.float32)
raw_mean, raw_std = X.mean().item(), X.std().item()
X_norm = (X - raw_mean) / raw_std
{"raw_mean": round(float(raw_mean), 4), "raw_std": round(float(raw_std), 4), "normalized_mean": round(float(X_norm.mean().item()), 4), "normalized_std": round(float(X_norm.std().item()), 4)}


### Interpretation
Normalization gives the optimizer a more stable numerical starting point, especially in deeper networks.


## 7.11.2 Augmentation in vision workflows

### Why It Matters
Augmentation can introduce translation, flipping, or rotation invariance. A simple rotation already changes the image while preserving the digit concept for many examples.


In [ ]:
import torch
from torchvision import transforms
from sklearn.datasets import load_digits

image = torch.tensor(load_digits().images[0] / 16.0, dtype=torch.float32).unsqueeze(0)
rotate = transforms.RandomRotation(degrees=(25, 25))
rotated = rotate(image)
{"original_sum": round(float(image.sum().item()), 3), "rotated_sum": round(float(rotated.sum().item()), 3)}


### Interpretation
Augmentation is a way of encoding which variations should still map to the same label.


## 7.11.3 Deeper convolution stacks

### Why It Matters
Adding depth lets the model compose local features into richer structures, but it also increases parameter count and optimization burden.


In [ ]:
import torch
from torch import nn

shallow = nn.Sequential(nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.Flatten(), nn.Linear(8 * 8 * 8, 10))
deep = nn.Sequential(nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.Flatten(), nn.Linear(16 * 8 * 8, 10))
{"shallow_params": sum(p.numel() for p in shallow.parameters()), "deep_params": sum(p.numel() for p in deep.parameters())}


### Interpretation
Depth can improve expressiveness, but it should be justified by the task and dataset size.


## 7.11.4 Transfer-learning intuition

### Why It Matters
Transfer learning usually means reusing a feature extractor and only retraining a smaller prediction head. Freezing parameters makes that setup explicit.


In [ ]:
import torch
from torchvision.models import resnet18

model = resnet18(weights=None)
for param in model.parameters():
    param.requires_grad = False
model.fc = torch.nn.Linear(model.fc.in_features, 10)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen = sum(p.numel() for p in model.parameters() if not p.requires_grad)
{"trainable_parameters": trainable, "frozen_parameters": frozen}


### Interpretation
No pretrained weights are needed to explain the idea: most of the network stays fixed while a small task head learns.


## 7.11.5 Confusion analysis for image tasks

### Why It Matters
Accuracy alone hides which classes are getting confused. A confusion matrix shows where the classifier is mixing labels.


In [ ]:
import torch
from torch import nn
from sklearn.datasets import load_digits
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split

digits = load_digits()
X = torch.tensor(digits.images / 16.0, dtype=torch.float32).unsqueeze(1)
y = torch.tensor(digits.target, dtype=torch.long)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=34, stratify=y)
model = nn.Sequential(nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), nn.Flatten(), nn.Linear(8 * 4 * 4, 10))
opt = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss()
for _ in range(15):
    opt.zero_grad()
    loss = loss_fn(model(X_train), y_train)
    loss.backward()
    opt.step()
with torch.no_grad():
    preds = model(X_test).argmax(dim=1).numpy()
cm = confusion_matrix(y_test.numpy(), preds)
cm[:3, :3].tolist()


### Interpretation
The confusion matrix tells you which classes deserve inspection, augmentation, or more examples.


## 7.11.6 Compare shallow versus improved CNN setup

### Why It Matters
This final subsection compares a shallow baseline with a slightly improved stack that adds depth and pooling.


In [ ]:
import copy
import torch
from torch import nn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

digits = load_digits()
X = torch.tensor(digits.images / 16.0, dtype=torch.float32).unsqueeze(1)
y = torch.tensor(digits.target, dtype=torch.long)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=35, stratify=y)

models = {
    "shallow": nn.Sequential(nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.Flatten(), nn.Linear(8 * 8 * 8, 10)),
    "improved": nn.Sequential(nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.Flatten(), nn.Linear(16 * 4 * 4, 10)),
}
results = {}
for name, model in models.items():
    opt = torch.optim.Adam(model.parameters(), lr=0.01)
    loss_fn = nn.CrossEntropyLoss()
    for _ in range(12):
        opt.zero_grad()
        loss = loss_fn(model(X_train), y_train)
        loss.backward()
        opt.step()
    with torch.no_grad():
        acc = (model(X_test).argmax(dim=1) == y_test).float().mean().item()
    results[name] = {"test_accuracy": round(float(acc), 3), "final_loss": round(float(loss.item()), 4)}
results


### Interpretation
The comparison keeps the data fixed and changes only the architectural practice, which is how model improvements should be judged.
